In [6]:
import numpy as np
from numpy.linalg import norm, solve, LinAlgError
import matplotlib.pyplot as plt

In [9]:
#These are the 5 case studies, functions f1 to f4. gradients are computed by the functions df1-df5 and Hessians by Hf1-Hf5
#The optimal value of a function in dimenson n can be obtained via x_opt(funcname,n), for example x_opt("f1",2)

# The Ellipsoid function
def f1(x,alpha=1000):
    w = alpha**(np.linspace(0,1,len(x)))
    return np.sum(w*x**2)
def df1(x,alpha=1000):
    w = alpha**(np.linspace(0,1,len(x))) 
    return 2*w*x
def Hf1(x,alpha=1000):
    w = alpha**(np.linspace(0,1,len(x)))
    return 2*np.diag(w)

#The Rosenbrock Banana Function
def f2(x):
    return (1-x[0])**2+100*(x[1]-x[0]**2)**2

def df2(x):
    (1-x[0])**2+100*(x[1]-x[0]**2)**2
    g0 = -2*(1-x[0])-400*(x[1]-x[0]**2)*x[0]
    g1 = 200*(x[1]-x[0]**2)
    return np.array([g0,g1])

def Hf2(x):
    H00 = 2 -400*x[1]+1200*x[0]**2
    H01 = -400*x[0]
    H11 = 200
    return np.array([[H00,H01],[H01,H11]])


# The Log-Ellipsoid Function
def f3(x,epsilon=10000.0,alpha=1000):
    return np.log1p(f1(x,alpha)*epsilon)

def df3(x,epsilon=10000,alpha=1000):
    fx=f1(x,alpha)
    return df1(x,alpha)/(fx+1.0/epsilon)

def Hf3(x,epsilon=10000,alpha=1000):
    fx=f1(x,alpha)
    dfx=df1(x,alpha)
    Hdx=Hf1(x,alpha)
    g = dfx/(fx+1.0/epsilon)
    return Hdx/(fx+1.0/epsilon)- np.outer(g,g)

# The Attractive-Sector Function - f(4)

def h(x,q):
    return (np.log(1+np.exp(-np.abs(q*x)))+np.maximum(q*x,0))/q
def dh(x,q):
    qx=np.clip(q*x,-100,100)
    return 1/(1+np.exp(-qx))

def Hh(x,q):
    return q*dh(x,q)*dh(-x,q)

def f4(x, q=1000):
    return np.sum(h(x,q)**2 + 100*h(-x,q)**2)

def df4(x, q=1000):
    return 2*h(x,q)*dh(x,q) - 200*h(-x,q)*dh(-x,q)

def Hf4(x, q=1000):
    h1 = dh(x,q)**2
    h2 = h(x,q)*Hh(x,q)
    f3 = -dh(-x,q)**2
    f4 = -h(-x,q)*Hh(-x,q)
    return np.diag(2*(h1+h2) - 200*(f3+f4))


# Sum of Powers Function
def f5(x):
    factor = 2*np.linspace(1,2,len(x))
    return np.sum(np.abs(x)**factor)

def df5(x):
    factor = 2*np.linspace(1,2,len(x))
    return np.sign(x)*factor*np.abs(x)**np.maximum(factor-1,0.0)

def Hf5(x):
    factor = 2*np.linspace(1,2,len(x))
    Hgx =factor*(factor-1)*np.abs(x)**np.maximum(factor-2,0.0)
    return np.diag(Hgx)


def x_opt(fname, dims):
    if fname=="f2":
        return np.ones(2)
    if fname=="f4":
        return np.ones(dims)*0.0019093310847216710905011
    return np.zeros(dims)

In [10]:
def solve_sphere_constrained(Q, b, Delta, tol=1e-8, max_iter=100):
    """
    Solve the quadratic minimization problem
      min (1/2 x^T Q x + b^T x)
      s.t. ||x||^2 <= Delta^2,
    using a bisection search on the Lagrange multiplier lambda.
    
    Returns:
      x_star: the computed solution,
      lambda_star: the final Lagrange multiplier,
      iter_count: number of bisection iterations,
      phi_val: final value of phi(lambda) (should be close to 0).
    """
    n = Q.shape[0]
    
    #  Compute the unconstrained minimizer: Q x + b = 0  =>  x = -Q^{-1}b.
    try:
        x_uncon = solve(Q, -b)
    except LinAlgError:
        raise ValueError("Matrix Q is singular or ill-conditioned.")

    # If the unconstrained solution is feasible, return it.
    if norm(x_uncon) <= Delta:
        return x_uncon, 0.0, 0, norm(x_uncon) - Delta

    # Define x(lambda) = -(Q + 2 lambda I)^{-1} b.
    def x_of_lambda(lam):
        return solve(Q + 2*lam*np.eye(n), -b)
    
    # Define phi(lambda) = ||x(lambda)|| - Delta.
    def phi(lam):
        return norm(x_of_lambda(lam)) - Delta

    # Step 1: Find an interval [lambda_low, lambda_high] where phi changes sign.
    lambda_low = 0.0
    lambda_high = 1.0
    while phi(lambda_high) > 0:
        lambda_high *= 2.0
        if lambda_high > 1e10:
            raise RuntimeError("Unable to bracket the root; problem may be ill-conditioned.")

    # Step 2: Bisection loop to find lambda such that ||x(lambda)|| ~= Delta.
    iter_count = 0
    lam_star = None
    x_star = None
    while iter_count < max_iter:
        lam_mid = 0.5 * (lambda_low + lambda_high)
        x_mid = x_of_lambda(lam_mid)
        # Update bracket based on the current norm.
        if norm(x_mid) > Delta:
            lambda_low = lam_mid
        else:
            lambda_high = lam_mid

        iter_count += 1
        
        # Check stopping criterion on lambda.
        if abs(lambda_high - lambda_low) < tol:
            lam_star = lam_mid
            x_star = x_mid
            break
    else:
        # If max_iter reached without convergence, return the last iterate.
        lam_star = lam_mid
        x_star = x_mid

    return x_star, lam_star, iter_count, phi(lam_star)

#Take KKT from last week
def check_KKT(Q, b, Delta, x, lam, tol=1e-6):
    """Check the KKT conditions for the trust-region subproblem.
    
    KKT conditions:
      1) Stationarity: (Q + 2*lam I)x + b = 0.
      2) Complementarity: lam * (||x||^2 - Delta^2) = 0.
      3) Primal feasibility: ||x||^2 <= Delta^2.
      4) Dual feasibility: lam >= 0.
    
    Returns a dictionary with residuals.
   """
    stationarity_resid = norm((Q + 2*lam*np.eye(Q.shape[0])).dot(x) + b)
    comp_resid = abs(lam * (norm(x)**2 - Delta**2))
    primal_feas = (norm(x)**2 - Delta**2) <= tol  # should be <= 0 (or near 0)
    dual_feas = lam >= -tol  # allow slight numerical negative drift
    
    return {
        'stationarity': stationarity_resid,
        'complementarity': comp_resid,
        'primal_feasibility': primal_feas,
        'dual_feasibility': dual_feas
    }
    
# -------------------------------
# Construct various test instances
# -------------------------------
def run_tests():
    n_runs = 100
    dims = 2
    alpha = 1000.0
    Delta = 1.0  # Trust-region radius.
    
    unconstrained_runs = 0
    active_runs = 0
    iterations_list = []
    kkt_stationarity_list = []
    tests = []
    
    for i in range(n_runs):
        # Random starting point x_k from [2,25]^dims.
        x_k = np.random.uniform(2, 25, dims)
        # For f1, the quadratic model at x_k:
        Q = Hf1(x_k, alpha)
        b = df1(x_k, alpha)
        # The unconstrained minimizer is p = -x_k.
        p_uncon = -x_k
        norm_uncon = norm(p_uncon)
    
    
    # Test Case 1: Unconstrained solution feasible
    # Q positive definite, unconstrained minimizer inside the ball.
    Q1 = np.array([[2.0, 0.0],
                   [0.0, 2.0]])
    b1 = np.array([-1.0, -1.0])
    Delta1 = 1.5  # norm(unconstrained) ~0.707, so feasible.
    tests.append(('Feasible Unconstrained', Q1, b1, Delta1))
    
    # Test Case 2: Unconstrained solution infeasible (boundary active)
    # Q positive definite, unconstrained minimizer outside the ball.
    Q2 = np.array([[2.0, 0.0],
                   [0.0, 2.0]])
    b2 = np.array([-2.0, -2.0])
    Delta2 = 1.0  # unconstrained minimizer would be [1,1] with norm ~1.414 > 1.
    tests.append(('Active Constraint', Q2, b2, Delta2))
    
    # Test Case 3: Indefinite Q (simple example)
    Q3 = np.array([[1.0, 0.0],
                   [0.0, -1.0]])
    b3 = np.array([0.5, 0.5])
    Delta3 = 1.0
    tests.append(('Indefinite Q', Q3, b3, Delta3))
    
    # Test Case 4: Negative definite Q
    Q4 = -np.array([[1.0, 0.0],
                    [0.0, 1.0]])
    b4 = np.array([1.0, 1.0])
    Delta4 = 0.8  # unconstrained minimizer is [1,1] (norm ~1.414), so constraint active.
    tests.append(('Negative Definite Q', Q4, b4, Delta4))
    
    print("Running Test Cases:\n")
    for name, Q, b, Delta in tests:
        print(f"Test: {name}")
        try:
            x_star, lam_star, iters, phi_val = solve_sphere_constrained(Q, b, Delta, tol=1e-8)
        except Exception as e:
            print("  Error:", e)
            continue

        kkt = check_KKT(Q, b, Delta, x_star, lam_star)
        print("  x* =", x_star)
        print("  lambda* =", lam_star)
        print("  ||x*|| =", norm(x_star))
        print("  Iterations =", iters)
        print("  phi(lambda*) =", phi_val)
        print("  KKT residuals:")
        for key, val in kkt.items():
            print(f"    {key}: {val}")
        print("-"*50)
        
# Run all tests
#run_tests()

In [11]:
# ----- Example 1: Direct test on f1 (Ellipsoid function) -----
def test_f1():
    # Choose a dimension and a point (for f1, the optimal is at 0 but we use a nonzero x_k)
    n = 5
    x_k = np.random.randn(n)  # arbitrary point
    Delta = 1.0  # trust-region radius
    
    # For f1(x) = sum(w*x^2) with w = alpha^(linspace(0,1,n))
    alpha = 1000.0
    # For a quadratic function, the model is exact.
    # Compute Hessian and gradient at x_k:
    Q = Hf1(x_k, alpha=alpha)   # Note: Hf1 returns 2*diag(w)
    b = df1(x_k, alpha=alpha)
    
    # Solve the sphere-constrained quadratic subproblem:
    p_star, lam_star, iters, phi_val = solve_sphere_constrained(Q, b, Delta, tol=1e-8)
    
    print("Test f1 (Ellipsoid Function):")
    print("x_k =", x_k)
    print("Computed step p* =", p_star)
    print("Lagrange multiplier =", lam_star)
    print("Iterations =", iters)
    print("||p*|| =", np.linalg.norm(p_star))
    print("phi(lambda*) =", phi_val)
    
    # Check KKT conditions:
    kkt = check_KKT(Q, b, Delta, p_star, lam_star)
    print("KKT residuals:")
    for key, val in kkt.items():
        print(f"  {key}: {val}")
    print("-"*50)

# ----- Example 2: Trust-region step for a non-quadratic function -----
def test_trust_region(f, df, Hf, fname, dims, Delta):
    # Let x_k be an arbitrary starting point. For demonstration, we take a random point.
    x_k = np.random.randn(dims)
    
    # Compute the quadratic model's data at x_k:
    g = df(x_k)
    H = Hf(x_k)
    
    # Our subproblem becomes: minimize m(p) = g^T p + 0.5 p^T H p, subject to ||p|| <= Delta.
    p_star, lam_star, iters, phi_val = solve_sphere_constrained(H, g, Delta, tol=1e-8)
    
    print(f"Trust-Region Test for {fname}:")
    print("x_k =", x_k)
    print("Computed step p* =", p_star)
    print("Lagrange multiplier =", lam_star)
    print("Iterations =", iters)
    print("||p*|| =", np.linalg.norm(p_star))
    print("phi(lambda*) =", phi_val)
    
    # KKT check for the quadratic model subproblem:
    kkt = check_KKT(H, g, Delta, p_star, lam_star)
    print("KKT residuals for the model subproblem:")
    for key, val in kkt.items():
        print(f"  {key}: {val}")
    print("-"*50)
    
    # Optionally, compare the model reduction:
    model_reduction = g.dot(p_star) + 0.5 * p_star.dot(H.dot(p_star))
    print("Model reduction =", model_reduction)
    print("="*50)

# Run the tests:
test_f1()

# You can similarly run trust-region tests on f2, f3, f4, f5.
# For example, testing on the Rosenbrock function (f2):
test_trust_region(f2, df2, Hf2, "f2 (Rosenbrock)", dims=2, Delta=0.5)


Test f1 (Ellipsoid Function):
x_k = [ 0.53823351 -0.82566394  0.98471     1.31288543  0.61685831]
Computed step p* = [-0.00472484  0.03916889 -0.21543914 -0.80300171 -0.55427222]
Lagrange multiplier = 112.91578892618418
Iterations = 34
||p*|| = 0.9999999999887847
phi(lambda*) = -1.1215250950158406e-11
KKT residuals:
  stationarity: 5.684341886080802e-14
  complementarity: 2.5327578180845463e-09
  primal_feasibility: True
  dual_feasibility: True
--------------------------------------------------
Trust-Region Test for f2 (Rosenbrock):
x_k = [ 1.05838735 -1.88546579]
Computed step p* = [-0.35342811  0.35367863]
Lagrange multiplier = 538.2975185737014
Iterations = 37
||p*|| = 0.5000000000005925
phi(lambda*) = 5.92526028242446e-13
KKT residuals for the model subproblem:
  stationarity: 0.0
  complementarity: 3.189552906932396e-10
  primal_feasibility: True
  dual_feasibility: True
--------------------------------------------------
Model reduction = -465.75938266963567
